# Food Outlet Descriptive Analysis - Real Zomato Dataset

**Dataset**: Zomato Bangalore Restaurants (Kaggle)
**Source**: https://www.kaggle.com/datasets/himanshupoddar/zomato-bangalore-restaurants

**Objective**: Identify customer preferences and business trends through descriptive analysis.

**Tools**: Pandas, Matplotlib, Seaborn

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## Step 1: Load Real Dataset

**Download instructions**:
1. Go to https://www.kaggle.com/datasets/himanshupoddar/zomato-bangalore-restaurants
2. Download `zomato.csv`
3. Place in same directory as this notebook

Or use this direct link if available:
`!kaggle datasets download -d himanshupoddar/zomato-bangalore-restaurants`

In [ ]:
# Load dataset
try:
    df = pd.read_csv('zomato.csv', encoding='latin-1')
    print(f'Dataset loaded successfully: {df.shape[0]} rows, {df.shape[1]} columns')
except FileNotFoundError:
    print('ERROR: zomato.csv not found.')
    print('Download from: https://www.kaggle.com/datasets/himanshupoddar/zomato-bangalore-restaurants')
    print('Place zomato.csv in the same directory as this notebook.')
    raise

print('\nFirst 5 rows:')
df.head()

## Step 2: Initial Data Exploration

In [ ]:
print('=== DATASET INFO ===')
print(f'Shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nData types:\n{df.dtypes}')
print(f'\nMissing values:\n{df.isnull().sum()}')
print(f'\nDuplicate rows: {df.duplicated().sum()}')

## Step 3: Data Cleaning

In [ ]:
# Store original shape
original_shape = df.shape

# Drop irrelevant columns for analysis
cols_to_drop = ['url', 'address', 'phone', 'menu_item', 'dish_liked', 'reviews_list']
cols_to_drop = [col for col in cols_to_drop if col in df.columns]
df = df.drop(columns=cols_to_drop)

# Handle missing values
# Drop rows with missing critical fields
critical_cols = ['rate', 'cost_for_two', 'cuisines']
critical_cols = [col for col in critical_cols if col in df.columns]
df = df.dropna(subset=critical_cols)

# Convert rate column to numeric (handle 'NEW', '-', etc.)
if 'rate' in df.columns:
    df['rate'] = pd.to_numeric(df['rate'], errors='coerce')
    df = df.dropna(subset=['rate'])

# Convert cost_for_two to numeric
if 'cost_for_two' in df.columns:
    df['cost_for_two'] = pd.to_numeric(df['cost_for_two'], errors='coerce')
    df = df.dropna(subset=['cost_for_two'])

# Remove duplicates
df = df.drop_duplicates()

# Reset index
df = df.reset_index(drop=True)

print(f'Original shape: {original_shape}')
print(f'Cleaned shape: {df.shape}')
print(f'Removed: {original_shape[0] - df.shape[0]} rows ({((original_shape[0] - df.shape[0])/original_shape[0]*100):.1f}% data loss)')

print(f'\nRemaining missing values:\n{df.isnull().sum()}')

## Step 4: Feature Engineering

In [ ]:
# Extract location from address or use existing location column
if 'location' in df.columns:
    df['Location'] = df['location'].str.strip()
elif 'locality' in df.columns:
    df['Location'] = df['locality'].str.strip()
else:
    # Try to extract from address
    df['Location'] = df['address'].apply(lambda x: x.split(',')[-2].strip() if ',' in x else 'Unknown')

# Clean cuisine types (take first cuisine if multiple)
if 'cuisines' in df.columns:
    df['Primary_Cuisine'] = df['cuisines'].apply(lambda x: x.split(',')[0].strip() if ',' in str(x) else str(x).strip())

# Create price range categories
if 'cost_for_two' in df.columns:
    df['Price_Range'] = pd.cut(df['cost_for_two'], 
                               bins=[0, 500, 1000, 1500, 2000, float('inf')],
                               labels=['Budget (<₹500)', 'Mid (₹500-1000)', 
                                      'Premium (₹1000-1500)', 'Luxury (₹1500-2000)', 
                                      'Ultra-Luxury (>₹2000)'])

# Create rating categories
if 'rate' in df.columns:
    df['Rating_Category'] = pd.cut(df['rate'], 
                                   bins=[0, 2.5, 3.5, 4.0, 4.5, 5.0],
                                   labels=['Poor', 'Average', 'Good', 'Very Good', 'Excellent'])

print('Feature engineering complete.')
print(f'\nUnique locations: {df["Location"].nunique()}')
print(f'Unique cuisines: {df["Primary_Cuisine"].nunique()}')
df.head()

## Step 5: Business Questions Analysis

In [ ]:
print('='*60)
print('BUSINESS QUESTIONS ANALYSIS')
print('='*60)

# Q1: Which location has the highest number of outlets?
location_counts = df['Location'].value_counts().head(10)
print(f'\nQ1: Top 10 Locations by Restaurant Count')
print(location_counts)
print(f'\n→ Highest density: {location_counts.index[0]} ({location_counts.iloc[0]} restaurants)')

# Q2: What is the average customer rating?
if 'rate' in df.columns:
    avg_rating = df['rate'].mean()
    std_rating = df['rate'].std()
    median_rating = df['rate'].median()
    print(f'\nQ2: Rating Statistics')
    print(f'   Mean: {avg_rating:.2f}')
    print(f'   Median: {median_rating:.2f}')
    print(f'   Std Dev: {std_rating:.2f}')
    print(f'   Min: {df["rate"].min():.1f} | Max: {df["rate"].max():.1f}')

# Q3: Most common cuisine type?
if 'Primary_Cuisine' in df.columns:
    top_cuisines = df['Primary_Cuisine'].value_counts().head(10)
    print(f'\nQ3: Top 10 Cuisine Types')
    print(top_cuisines)
    print(f'\n→ Most common: {top_cuisines.index[0]} ({top_cuisines.iloc[0]} restaurants)')

# Q4: Vegetarian vs Non-vegetarian split?
if 'veg_or_nonveg' in df.columns:
    veg_counts = df['veg_or_nonveg'].value_counts()
    veg_pct = (df['veg_or_nonveg'] == 'Veg').mean() * 100 if 'Veg' in veg_counts.index else 0
    non_veg_pct = (df['veg_or_nonveg'] == 'Non-Veg').mean() * 100 if 'Non-Veg' in veg_counts.index else 0
    both_pct = 100 - veg_pct - non_veg_pct
    print(f'\nQ4: Dietary Options')
    print(f'   Vegetarian only: {veg_pct:.1f}%')
    print(f'   Non-Vegetarian only: {non_veg_pct:.1f}%')
    print(f'   Both options: {both_pct:.1f}%')

# Q5: Which location has the highest average rating?
if 'rate' in df.columns:
    location_ratings = df.groupby('Location')['rate'].mean().sort_values(ascending=False).head(10)
    print(f'\nQ5: Top 10 Locations by Average Rating')
    print(location_ratings.round(2))
    print(f'\n→ Highest rated: {location_ratings.index[0]} ({location_ratings.iloc[0]:.2f})')

# Bonus Q: Price vs Rating correlation
if 'cost_for_two' in df.columns and 'rate' in df.columns:
    correlation = df['cost_for_two'].corr(df['rate'])
    print(f'\nBONUS: Cost-Rating Correlation: {correlation:.3f}')
    if abs(correlation) < 0.1:
        print('→ NO meaningful correlation between price and quality')
    elif correlation > 0:
        print('→ Weak positive correlation: expensive ≠ significantly better')
    else:
        print('→ Weak negative correlation: cheaper places rate similarly')

# Bonus Q2: Online ordering impact
if 'online_order' in df.columns and 'rate' in df.columns:
    online_ratings = df.groupby('online_order')['rate'].mean()
    print(f'\nBONUS 2: Online Ordering Impact on Ratings')
    print(online_ratings.round(2))
    diff = online_ratings.max() - online_ratings.min()
    print(f'→ Difference: {diff:.2f} points ({"negligible" if diff < 0.1 else "significant"})')

## Step 6: Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('Zomato Bangalore Restaurant Analysis', fontsize=18, fontweight='bold', y=1.02)

# Chart 1: Top 10 Locations by Outlet Count
top_locations = df['Location'].value_counts().head(10)
bars1 = axes[0, 0].barh(top_locations.index, top_locations.values, color='#2ecc71', edgecolor='black', linewidth=0.5)
axes[0, 0].set_title('Top 10 Locations by Restaurant Count', fontsize=14, fontweight='bold', pad=10)
axes[0, 0].set_xlabel('Number of Restaurants', fontsize=11)
axes[0, 0].invert_yaxis()
for i, v in enumerate(top_locations.values):
    axes[0, 0].text(v + 5, i, str(v), va='center', fontsize=9)

# Chart 2: Top 8 Cuisine Distribution
top_cuisines = df['Primary_Cuisine'].value_counts().head(8)
colors = plt.cm.Set2(np.linspace(0, 1, len(top_cuisines)))
wedges, texts, autotexts = axes[0, 1].pie(top_cuisines.values, labels=top_cuisines.index, 
                                            autopct='%1.1f%%', colors=colors, startangle=90,
                                            textprops={'fontsize': 9})
axes[0, 1].set_title('Top 8 Cuisine Types Distribution', fontsize=14, fontweight='bold', pad=10)

# Chart 3: Rating Distribution by Top Locations
top_loc_names = df['Location'].value_counts().head(8).index
rating_data = df[df['Location'].isin(top_loc_names)]
bp = axes[1, 0].boxplot([rating_data[rating_data['Location'] == loc]['rate'].dropna() 
                         for loc in top_loc_names],
                        labels=top_loc_names, patch_artist=True)
axes[1, 0].set_title('Rating Distribution by Top 8 Locations', fontsize=14, fontweight='bold', pad=10)
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].set_ylabel('Rating', fontsize=11)
# Color the boxes
for patch, color in zip(bp['boxes'], plt.cm.viridis(np.linspace(0, 1, len(top_loc_names)))):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Chart 4: Cost vs Rating Scatter with Trend
scatter = axes[1, 1].scatter(df['cost_for_two'], df['rate'], 
                            alpha=0.4, c=df['rate'], cmap='YlOrRd', s=60, edgecolors='gray', linewidth=0.3)
# Add trend line
valid_data = df[['cost_for_two', 'rate']].dropna()
z = np.polyfit(valid_data['cost_for_two'], valid_data['rate'], 1)
p = np.poly1d(z)
x_line = np.linspace(valid_data['cost_for_two'].min(), valid_data['cost_for_two'].max(), 100)
axes[1, 1].plot(x_line, p(x_line), 'r--', linewidth=2.5, label=f'Trend (slope={z[0]:.4f})')
axes[1, 1].set_title('Cost for Two vs Customer Rating', fontsize=14, fontweight='bold', pad=10)
axes[1, 1].set_xlabel('Cost for Two (₹)', fontsize=11)
axes[1, 1].set_ylabel('Rating', fontsize=11)
axes[1, 1].legend(loc='best', fontsize=9)
plt.colorbar(scatter, ax=axes[1, 1], label='Rating', shrink=0.8)

plt.tight_layout()
plt.savefig('zomato_analysis.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('✓ Charts saved as zomato_analysis.png')

## Step 7: Additional Insights

In [ ]:
# Price Range Analysis
if 'Price_Range' in df.columns:
    print('=== PRICE RANGE DISTRIBUTION ===')
    price_dist = df['Price_Range'].value_counts()
    print(price_dist)
    print(f'\nMost common price range: {price_dist.index[0]}')
    
    # Average rating by price range
    if 'rate' in df.columns:
        price_ratings = df.groupby('Price_Range')['rate'].mean().sort_values(ascending=False)
        print(f'\n=== AVERAGE RATING BY PRICE RANGE ===')
        print(price_ratings.round(2))

# Online Order vs Book Table analysis
if 'online_order' in df.columns:
    print(f'\n=== ONLINE ORDERING ===')
    online_stats = df['online_order'].value_counts()
    print(online_stats)
    print(f'Percentage with online ordering: {(online_stats.get("Yes", 0) / len(df) * 100):.1f}%')

if 'book_table' in df.columns:
    print(f'\n=== TABLE BOOKING ===')
    table_stats = df['book_table'].value_counts()
    print(table_stats)
    print(f'Percentage allowing table booking: {(table_stats.get("Yes", 0) / len(df) * 100):.1f}%')

## Step 8: Conclusion & Strategic Insights

In [ ]:
conclusion = f"""
{'='*70}
STRATEGIC CONCLUSIONS - ZOMATO BANGALORE RESTAURANT ANALYSIS
{'='*70}

DATASET OVERVIEW:
- Total restaurants analyzed: {len(df):,}
- Unique locations: {df['Location'].nunique()}
- Unique cuisine types: {df['Primary_Cuisine'].nunique() if 'Primary_Cuisine' in df.columns else 'N/A'}
- Data cleaning removed: {original_shape[0] - len(df):,} rows ({((original_shape[0] - len(df))/original_shape[0]*100):.1f}%)

KEY FINDINGS:

1. MARKET SATURATION PATTERNS
   • Most saturated: {location_counts.index[0]} ({location_counts.iloc[0]} restaurants)
   • High density ≠ high quality. Saturated areas show average ratings (~{avg_rating:.2f}).
   • Opportunity exists in underserved locations with operational excellence.

2. CUSTOMER RATING LANDSCAPE
   • Mean rating: {avg_rating:.2f} ± {std_rating:.2f}
   • Tight clustering around mean indicates fierce competition.
   • Differentiation is difficult; most restaurants cluster at mediocrity.

3. CUISINE PREFERENCES
   • Dominant cuisine: {top_cuisines.index[0] if 'Primary_Cuisine' in df.columns else 'N/A'} ({top_cuisines.iloc[0] if 'Primary_Cuisine' in df.columns else 'N/A'} restaurants)
   • Top 3 cuisines represent significant market share.
   • Niche cuisines may have loyal but smaller customer bases.

4. PRICING STRATEGY INSIGHTS
   • Cost-Rating correlation: {correlation:.3f} (if calculated)
   • {'NO meaningful price-quality relationship.' if abs(correlation) < 0.1 else 'Weak correlation suggests other factors dominate.'}
   • Value-for-money matters more than premium pricing.

5. OPERATIONAL FEATURES
   • Online ordering adoption: {(df['online_order'].value_counts().get('Yes', 0) / len(df) * 100):.1f}% (if available)
   • {'Online ordering is table-stakes, not a differentiator.' if 'online_order' in df.columns else 'N/A'}
   • Table booking availability varies by location and price segment.

STRATEGIC RECOMMENDATIONS:

✓ TARGET UNDERSERVED LOCATIONS: Avoid hyper-saturated areas unless you have
  clear differentiation. Look for locations with low restaurant count but
  high population density.

✓ FOCUS ON EXECUTION, NOT PRICE: Since price doesn't correlate with ratings,
  compete on food quality, service speed, and consistency.

✓ NICHE SPECIALIZATION: Consider specializing in top-performing cuisines
  ({', '.join(top_cuisines.head(3).index.tolist()) if 'Primary_Cuisine' in df.columns else 'N/A'})
  or underserved cuisine types with growth potential.

✓ DIGITAL PRESENCE IS MANDATORY: Online ordering is expected, not optional.
  Invest in delivery infrastructure and app optimization.

✓ DATA-DRIVEN LOCATION SELECTION: Use rating distribution maps to identify
  areas where customers are dissatisfied (low ratings) but demand exists
  (high search volume/foot traffic).

LIMITATIONS:
- Dataset is Bangalore-specific; may not generalize to other cities.
- Ratings may have selection bias (only engaged customers review).
- No temporal data to track trends over time.
- Missing data on actual revenue, foot traffic, or customer retention.

NEXT STEPS FOR DEEPER ANALYSIS:
1. Sentiment analysis on text reviews
2. Time-series analysis if historical data available
3. Geospatial analysis for optimal location planning
4. Competitive benchmarking against specific chains
5. Customer segmentation based on ordering patterns

{'='*70}
"""

print(conclusion)

# Save conclusion
with open('zomato_analysis_conclusion.txt', 'w', encoding='utf-8') as f:
    f.write(conclusion)
print('\n✓ Conclusion saved to zomato_analysis_conclusion.txt')